<a href="https://colab.research.google.com/github/Vdmtx/FBNeo-Android/blob/main/gerador_de_imagens_nexus_%C3%A9ter_4_varia%C3%A7%C3%B5es.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ✅ ETAPA 1 – INSTALAÇÃO DE DEPENDÊNCIAS
!pip install --quiet --upgrade diffusers transformers accelerate safetensors rembg onnxruntime torch torchvision pillow gradio

# ✅ ETAPA 2 – IMPORTAÇÕES E CONFIGURAÇÕES
import torch, os, warnings
from PIL import Image
from io import BytesIO
from datetime import datetime
from diffusers import StableDiffusionPipeline, StableDiffusionXLPipeline, UniPCMultistepScheduler
from rembg import remove
import gradio as gr

warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Dispositivo: {device}")

if device.type == "cuda":
    print(f"🚀 GPU: {torch.cuda.get_device_name(0)}")

# ✅ FUNÇÃO DE REMOÇÃO DE FUNDO
def remover_fundo(imagem):
    try:
        if imagem is None: return None
        buffer = BytesIO()
        imagem.save(buffer, format="PNG")
        removida = remove(buffer.getvalue())
        return Image.open(BytesIO(removida))
    except Exception as e:
        print(f"❌ Erro na remoção de fundo: {e}")
        return imagem

# ✅ FUNÇÕES DE CARREGAMENTO DE MODELOS
modelo_cache = {}

def carregar_modelo(nome, tipo="SDXL"):
    if nome in modelo_cache:
        return modelo_cache[nome]

    print(f"📦 Carregando modelo: {nome}")
    if tipo == "SDXL":
        pipe = StableDiffusionXLPipeline.from_pretrained(
            nome,
            torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
            use_safetensors=True,
            variant="fp16" if device.type == "cuda" else None
        )
    else:
        pipe = StableDiffusionPipeline.from_pretrained(
            nome,
            torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
            use_safetensors=True,
            safety_checker=None,
            requires_safety_checker=False
        )

    if device.type == "cuda":
        pipe.enable_model_cpu_offload()
        pipe.enable_attention_slicing()

    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
    pipe.to(device)
    modelo_cache[nome] = pipe
    return pipe

# ✅ GERAÇÃO DE IMAGENS
def gerar(prompt, negativo, steps, largura, altura, modelo, imagem_upload, remover_bg, num_var):
    if not prompt.strip(): return None, "⚠️ Prompt vazio."

    # Remoção de fundo (opcional)
    imagem_ref = None
    if imagem_upload and remover_bg:
        imagem_ref = remover_fundo(imagem_upload)

    nome_modelo = "stabilityai/stable-diffusion-xl-base-1.0" if modelo == "SDXL" else "runwayml/stable-diffusion-v1-5"
    tipo = "SDXL" if modelo == "SDXL" else "SD1.5"
    pipe = carregar_modelo(nome_modelo, tipo)

    params = {
        "prompt": prompt,
        "negative_prompt": negativo,
        "num_inference_steps": steps,
        "height": altura,
        "width": largura,
        "num_images_per_prompt": num_var
    }

    with torch.inference_mode():
        print("🎨 Gerando imagens...")
        resultado = pipe(**params)

    imagens = resultado.images
    nomes = []
    for i, img in enumerate(imagens):
        nome = f"nexus_eter_{datetime.now().strftime('%H%M%S')}_{i+1}.png"
        img.save(nome)
        nomes.append(nome)

    galeria = [Image.open(n) for n in nomes]
    return galeria, f"✅ {len(galeria)} imagem(ns) gerada(s)!"

# ✅ INTERFACE GRADIO
with gr.Blocks(theme=gr.themes.Soft()) as interface:
    gr.Markdown("## ✨ Nexus Éter Generator – Fase 1 (Colab)\nGeração de imagens com galeria, múltiplas variações e remoção de fundo.")

    with gr.Row():
        with gr.Column(scale=2):
            prompt = gr.Textbox(label="🎯 Prompt", lines=3)
            negativo = gr.Textbox(label="🚫 Prompt Negativo", value="lowres, blurry, bad anatomy", lines=2)

            modelo = gr.Radio(["SDXL", "SD1.5"], value="SDXL", label="🧠 Modelo")

            steps = gr.Slider(10, 60, value=30, step=1, label="🔁 Steps")
            num_var = gr.Slider(1, 4, value=2, step=1, label="🖼️ Variações")

            largura = gr.Slider(512, 1024, step=64, value=768, label="📐 Largura")
            altura = gr.Slider(512, 1024, step=64, value=768, label="📏 Altura")

            imagem_upload = gr.Image(label="🖼️ Upload para Referência (opcional)", type="pil")
            remover_bg = gr.Checkbox(label="❎ Remover fundo (imagem enviada)", value=False)

            botao = gr.Button("🚀 Gerar")

        with gr.Column(scale=1):
            galeria = gr.Gallery(label="🎨 Resultado", columns=2, object_fit="contain", height=512)
            status = gr.Textbox(label="📊 Status", interactive=False)

    botao.click(
        fn=gerar,
        inputs=[prompt, negativo, steps, largura, altura, modelo, imagem_upload, remover_bg, num_var],
        outputs=[galeria, status]
    )

interface.launch(share=True)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.7/41.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.2/821.2 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 126.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 93.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 55.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 57.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.2/158.2 MB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.6